In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
# =========================
# 1. Imports
# =========================
import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# =========================
# 2. LOAD DATASET
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/svm-devignx26"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        path = os.path.join(root, f)
        print("Trying:", path)
        if f.endswith(".csv"):
            df = pd.read_csv(path)
            print("✅ Loaded:", path)
            break
    if df is not None:
        break

if df is None:
    raise Exception("❌ Dataset not found")

print("\n📊 Shape:", df.shape)
print("📌 Columns:", df.columns)

# =========================
# 3. PREPROCESS
# =========================
df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)

# =========================
# 4. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].astype(str),
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# =========================
# 5. TF-IDF FEATURES
# =========================
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r'\b\w+\b'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# =========================
# 6. SVM MODEL + CALIBRATION
# =========================
svm = LinearSVC(class_weight="balanced", C=1.0)

# Calibration gives predict_proba
model = CalibratedClassifierCV(svm, method='sigmoid', cv=3)
model.fit(X_train_vec, y_train)

# =========================
# 7. THRESHOLD TUNING (SAFE)
# =========================
probs = model.predict_proba(X_test_vec)[:, 1]

best_f1 = 0
best_t = 0.5
best_preds = None

print("\n🔍 Threshold tuning:")

for t in np.arange(0.4, 0.7, 0.05):
    preds = (probs > t).astype(int)
    report_temp = classification_report(y_test, preds, output_dict=True)

    recall_0 = report_temp['0']['recall']
    recall_1 = report_temp['1']['recall']
    f1 = report_temp['1']['f1-score']

    print(f"t={t:.2f} → F1={f1:.4f}, R0={recall_0:.2f}, R1={recall_1:.2f}")

    # relaxed constraint to avoid no-selection
    if recall_0 > 0.2 and recall_1 > 0.2:
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_preds = preds

# fallback if nothing selected
if best_preds is None:
    print("\n⚠️ No valid threshold found → using default 0.5")
    best_preds = (probs > 0.5).astype(int)

print("\nBest threshold:", best_t)

# =========================
# 8. FINAL EVALUATION
# =========================
final_preds = best_preds

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 9. SAVE RESULTS
# =========================
label_key = [k for k in report.keys() if str(k).startswith("1")][0]

results = {
    "Model": "SVM",
    "Dataset": "Devign",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/svm_devign_results.csv", index=False)

print("\n✅ Results saved in /kaggle/working/")

Trying: /kaggle/input/datasets/nikunjnawal009/svm-devignx26/Devignx_validation.csv
✅ Loaded: /kaggle/input/datasets/nikunjnawal009/svm-devignx26/Devignx_validation.csv

📊 Shape: (2732, 2)
📌 Columns: Index(['code', 'label'], dtype='object')

🔍 Threshold tuning:
t=0.40 → F1=0.6147, R0=0.16, R1=0.93
t=0.45 → F1=0.3660, R0=0.77, R1=0.29
t=0.50 → F1=0.0329, R0=1.00, R1=0.02
t=0.55 → F1=0.0000, R0=1.00, R1=0.00
t=0.60 → F1=0.0000, R0=1.00, R1=0.00
t=0.65 → F1=0.0000, R0=1.00, R1=0.00

Best threshold: 0.45

✅ Accuracy: 0.5630712979890311

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.77      0.67       309
           1       0.50      0.29      0.37       238

    accuracy                           0.56       547
   macro avg       0.54      0.53      0.52       547
weighted avg       0.55      0.56      0.54       547


✅ Results saved in /kaggle/working/


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m